In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-07 02:20:16.766504: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-07 02:20:17.436829: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 3,
  "chunk_size": 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 5,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-07 02:20:18,563 [DEBUG] [Rain] Rain is initialized
2023-07-07 02:20:18,565 [DEBUG] [Provisioner] Creating coordinator
2023-07-07 02:20:18,566 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-07 02:20:18,567 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-07 02:20:18,567 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-07 02:20:18,568 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 02:20:18,569 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-07 02:20:18,570 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-07 02:20:18,581 [INFO] [Provisioner] provisioner is serving
2023-07-07 02:20:18,582 [DEBUG] [Provisioner] Starting coordinator
2023-07-07 02:20:18,583 [INFO] [Coordinator] coordinator is serving
2023-07-07 02:20:18,584 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-07 02:20:18,587 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 02:20:18,588 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-07 02:20:18,589 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-07 02:20:18,590 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50151/
2023-07-07 02:20:18,592 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-07 02:20:18,593 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50152/
2023-07-07 02:20:18,595 [INFO] [Worker_50152] Worker is running on port: 50152
2023-0

Epoch 1/5
Epoch 1/5
Epoch 1/5
157/157 [==============================] - 3s 8ms/step - loss: 0.7032 - accuracy: 0.7816
Epoch 2/5
157/157 [==============================] - 3s 8ms/step - loss: 0.7047 - accuracy: 0.7789
Epoch 2/5
157/157 [==============================] - 1s 8ms/step - loss: 0.3140 - accuracy: 0.9060
Epoch 3/5
157/157 [==============================] - 1s 8ms/step - loss: 0.3033 - accuracy: 0.9089
Epoch 3/5
157/157 [==============================] - 1s 8ms/step - loss: 0.2348 - accuracy: 0.9301
Epoch 4/5
157/157 [==============================] - 1s 8ms/step - loss: 0.2307 - accuracy: 0.9301
Epoch 4/5
157/157 [==============================] - 1s 8ms/step - loss: 0.1915 - accuracy: 0.9431
Epoch 5/5
157/157 [==============================] - 1s 8ms/step - loss: 0.1956 - accuracy: 0.9405
Epoch 5/5
Epoch 5/5
157/157 [==============================] - 1s 8ms/step - loss: 0.1667 - accuracy: 0.9503


2023-07-07 02:20:28,665 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3


sending data to divider
157/157 [==============================] - 1s 8ms/step - loss: 0.1706 - accuracy: 0.9483


2023-07-07 02:20:28,668 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-07 02:20:28,681 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
2023-07-07 02:20:28,683 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
2023-07-07 02:20:28,683 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-07 02:20:28,684 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2


...

2023-07-07 02:20:45,767 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 1.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 1.


157/157 [==============================] - 1s 5ms/step - loss: 0.1192 - accuracy: 0.9653
Epoch 3/5
157/157 [==============================] - 1s 4ms/step - loss: 0.0992 - accuracy: 0.9681
Epoch 4/5
157/157 [==============================] - 1s 4ms/step - loss: 0.0990 - accuracy: 0.9695
Epoch 5/5
157/157 [==============================] - 1s 3ms/step - loss: 0.0897 - accuracy: 0.9721


2023-07-07 02:20:47,825 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3
2023-07-07 02:20:47,827 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3


sending data to divider


2023-07-07 02:20:47,890 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-07 02:20:47,898 [DEBUG] [DeepLearning] Asynchronous update is done by worker 3
DEBUG:DeepLearning:Asynchronous update is done by worker 3
2023-07-07 02:20:47,927 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 3.
DEBUG:DeepLearning:Iteration 3/3 complete for worker 3.
2023-07-07 02:20:47,929 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-07 02:20:47,931 [DEBUG] [Divider] Divider stopped serving
DEBUG:Divider:Divider stopped serving


In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0861 - accuracy: 0.9769

Test accuracy: 97.7%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-07 02:20:48,270 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-07 02:20:48,271 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-07 02:20:48,273 [INFO] [Coordinator] coordinator is serving
INFO:Coordinator:coordinator is serving
2023-07-07 02:20:48,274 [DEBUG] [Coordinator] sending the num of workers to the provisioner
DEBUG:Coordinator:sending the num of workers to the provisioner
2023-07-07 02:20:48,276 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-07 02:20:48,278 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
DEBUG:Coordinator:sent Success receiving the number of workers to the provisioner
2023-07-07 02:20:48,279 [DEBUG] [LocalProvisioner] Creating 3 workers
DEBUG:LocalProvisioner:Creating 3 

Epoch 1/5
Epoch 1/5
Epoch 1/5
157/157 [==============================] - 3s 9ms/step - loss: 0.1126 - accuracy: 0.9661
Epoch 2/5
157/157 [==============================] - 3s 9ms/step - loss: 0.1086 - accuracy: 0.9689
Epoch 2/5
157/157 [==============================] - 3s 9ms/step - loss: 0.1081 - accuracy: 0.9664
Epoch 2/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0949 - accuracy: 0.9719
Epoch 3/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0883 - accuracy: 0.9732
Epoch 3/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0981 - accuracy: 0.9692
Epoch 3/5
157/157 [==============================] - 1s 8ms/step - loss: 0.0888 - accuracy: 0.9739
Epoch 4/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0794 - accuracy: 0.9753
Epoch 5/5
157/157 [==============================] - 1s 8ms/step - loss: 0.0739 - accuracy: 0.9772
Epoch 5/5
157/157 [==============================] - 1s 9ms/step - loss: 0.0768 - accurac

2023-07-07 02:20:59,133 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-07 02:20:59,137 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2


sending data to divider
143/157 [==========================>...] - ETA: 0s - loss: 0.0786 - accuracy: 0.9744

2023-07-07 02:20:59,156 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-07 02:20:59,159 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


...

157/157 [==============================] - 1s 7ms/step - loss: 0.0525 - accuracy: 0.9823


2023-07-07 02:21:22,919 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1


sending data to divider


DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker1
2023-07-07 02:21:22,920 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_3_trained.pkl from worker1
2023-07-07 02:21:22,978 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
2023-07-07 02:21:24,275 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker3
2023-07-07 02:21:24,276 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_3_trained.pk

sending data to divider


In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0646 - accuracy: 0.9827

Test accuracy: 98.3%
